# Multi-Seed Training — ByT5 for Arabic ITSM Classification

Runs ByT5-base (EXP-008) across 3 seeds to get stable mean ± std for Section 7.

**Estimated time**: ~1.5 GPU hours (~30 min per run × 3 seeds)

**Configuration**: ByT5 L1+L2+L3 (seeds: 42, 123, 456)

**Key difference from BERT-family**: ByT5 uses byte-level tokenization (`--max-length 512`)  
and mean pooling instead of [CLS]. The classifier auto-detects this from the model name.

In [ ]:
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

In [ ]:
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
from huggingface_hub import snapshot_download

print('Downloading google/byt5-base to local disk...')
BYT5_PATH = snapshot_download(
    'google/byt5-base',
    local_dir='/kaggle/working/model_cache/byt5'
)
print('ByT5-base ready at:', BYT5_PATH)

In [ ]:
from pathlib import Path

METRICS_DIR = Path('results/metrics/multi_seed_byt5')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

CONFIG = {
    'name': 'byt5_l1l2l3',
    'model': BYT5_PATH,
    'flags': '--tasks l1 l2 l3',
    'task_label': 'l1_l2_l3',
    'epochs': 10,
    'extra': '--max-length 512',
}

print('Runs to execute:')
for seed in SEEDS:
    print(f'  {CONFIG["name"]} seed={seed}')

In [ ]:
import json, time

ipy = get_ipython()
run_log = []

for seed in SEEDS:
    run_key = CONFIG['name'] + '_seed' + str(seed)
    out_metrics_path = METRICS_DIR / (run_key + '_metrics.json')

    if out_metrics_path.exists():
        print('[SKIP]', run_key, '-- already exists')
        continue

    seed_dir = 'models/seed_' + str(seed)
    checkpoint_dir = Path(seed_dir) / ('marbert_' + CONFIG['task_label'] + '_best')

    cmd = ' '.join([
        'python scripts/train.py', CONFIG['flags'],
        '--model', CONFIG['model'],
        '--epochs', str(CONFIG['epochs']),
        '--seed', str(seed),
        '--output-dir', seed_dir,
        '--data-dir data/processed',
        CONFIG['extra'],
    ])

    print('')
    print('[RUN]', run_key)
    print('  CMD:', cmd)
    t0 = time.time()
    ipy.system(cmd)
    elapsed = time.time() - t0
    print('  Time: %.1f min' % (elapsed/60,))

    test_metrics_file = checkpoint_dir / 'test_metrics.json'
    if test_metrics_file.exists():
        with open(test_metrics_file) as fp:
            test_metrics = json.load(fp)
        result = {'config': CONFIG['name'], 'seed': seed, 'metrics': test_metrics}
        with open(out_metrics_path, 'w') as fp:
            json.dump(result, fp, indent=2)
        print('  Saved:', out_metrics_path)
        run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed/60})
    else:
        print('  WARNING: test_metrics.json not found at', test_metrics_file)
        run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed/60})

print('')
print('=== Run log ===')
for r in run_log:
    print(' ', r['run'] + ':', r['status'], '(%.1f min)' % r['time_min'])

In [ ]:
# Summarize results across seeds
import json, numpy as np
from pathlib import Path

results = {}
for seed in SEEDS:
    run_key = f'byt5_l1l2l3_seed{seed}'
    path = Path('results/metrics/multi_seed_byt5') / (run_key + '_metrics.json')
    if path.exists():
        with open(path) as f:
            results[seed] = json.load(f)['metrics']

print(f'Seeds completed: {list(results.keys())}')
print()
for task in ['l1', 'l2', 'l3']:
    key = f'{task}_macro_f1'
    vals = [results[s][key] * 100 for s in results if key in results[s]]
    if vals:
        print(f'ByT5 {task.upper()} macro-F1: {np.mean(vals):.2f} ± {np.std(vals):.2f}%  (seeds: {list(results.keys())})')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/byt5_multi_seed_results', 'zip',
                    'results/metrics', 'multi_seed_byt5')
print('Download: /kaggle/working/byt5_multi_seed_results.zip')